# 03 Semantic Partition And Evaluation

Assignment step 6: partition global `V` into measures `M`, dimension names `N`, dimension values `A`, units `U`, plus `other_ambiguous` for unsafe cases.


In [1]:
import csv
import json
from pathlib import Path

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def read_json(relative_path: str):
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"missing": str(path)}


def csv_rows(relative_path: str, limit: int | None = None):
    csv.field_size_limit(2_147_483_647)
    path = ROOT / relative_path
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    return rows if limit is None else rows[:limit]


def csv_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    if not path.exists():
        return None
    return len(csv_rows(relative_path))


def parquet_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    return pq.read_table(path).num_rows if path.exists() else None

## Required Output Counts


In [2]:
{
    "M measures.csv": csv_count("outputs/measures.csv"),
    "N dimension_names.csv": csv_count("outputs/dimension_names.csv"),
    "A dimension_values.csv": csv_count("outputs/dimension_values.csv"),
    "U units.csv": csv_count("outputs/units.csv"),
    "other_ambiguous.csv": csv_count("outputs/other_ambiguous.csv"),
}

{'M measures.csv': 2894,
 'N dimension_names.csv': 359,
 'A dimension_values.csv': 607,
 'U units.csv': 261,
 'other_ambiguous.csv': 9114}

## Quality Evaluation


In [3]:
metrics = read_json("report/classification_metrics.json")
{
    "variant": metrics.get("variant"),
    "gold_status": metrics.get("gold_status"),
    "agreement": metrics.get("agreement"),
    "overall": metrics.get("metrics", {}),
    "validation": metrics.get("validation_metrics", {}),
    "final_test": metrics.get("final_test_metrics", {}),
}

{'variant': None,
 'gold_status': 'available',
 'agreement': {'random_sample': {'cohen_kappa': 1.0,
   'disagreements': [],
   'raw_agreement': 1.0,
   'reviewed_count': 50,
   'status': 'available'},
  'targeted_reclaim': {'cohen_kappa': 1.0,
   'disagreements': [],
   'raw_agreement': 1.0,
   'reviewed_count': 25,
   'status': 'available'}},
 'overall': {'accuracy': 0.6790945406125166,
  'confusion_matrix': {'dimension_name': {'dimension_name': 84,
    'dimension_value': 0,
    'measure': 0,
    'other_ambiguous': 2,
    'unit': 0},
   'dimension_value': {'dimension_name': 0,
    'dimension_value': 59,
    'measure': 0,
    'other_ambiguous': 72,
    'unit': 0},
   'measure': {'dimension_name': 0,
    'dimension_value': 0,
    'measure': 22,
    'other_ambiguous': 134,
    'unit': 0},
   'other_ambiguous': {'dimension_name': 0,
    'dimension_value': 15,
    'measure': 0,
    'other_ambiguous': 326,
    'unit': 0},
   'unit': {'dimension_name': 0,
    'dimension_value': 8,
    'measu

## Example Classified Terms


In [4]:
def compact_rows(relative_path: str, limit: int = 3):
    keep = ["term_id", "term", "label", "category", "confidence", "table_count", "occurrence_count"]
    rows = []
    for row in csv_rows(relative_path, limit):
        rows.append({key: row.get(key) for key in keep if key in row})
    return rows


{
    "measures": compact_rows("outputs/measures.csv"),
    "dimension_names": compact_rows("outputs/dimension_names.csv"),
    "dimension_values": compact_rows("outputs/dimension_values.csv"),
    "units": compact_rows("outputs/units.csv"),
    "other_ambiguous": compact_rows("outputs/other_ambiguous.csv"),
}

{'measures': [{'term_id': 'term_f92aa0607c2b5c538059',
   'category': 'measure',
   'confidence': '0.88'},
  {'term_id': 'term_ae1f4ef44ee128d00b40',
   'category': 'measure',
   'confidence': '0.88'},
  {'term_id': 'term_290bbaa73df30999a339',
   'category': 'measure',
   'confidence': '0.88'}],
 'dimension_names': [{'term_id': 'term_74ea855f131291a5db83',
   'category': 'dimension_name',
   'confidence': '0.98'},
  {'term_id': 'term_4ae69896984e2ec32e17',
   'category': 'dimension_name',
   'confidence': '0.98'},
  {'term_id': 'term_69d6d8044fe022b058e1',
   'category': 'dimension_name',
   'confidence': '0.98'}],
 'dimension_values': [{'term_id': 'term_f2605a3223c3aeb9adfc',
   'category': 'dimension_value',
   'confidence': '0.5108299871912039'},
  {'term_id': 'term_67aee9b0ccb060e573e4',
   'category': 'dimension_value',
   'confidence': '0.94'},
  {'term_id': 'term_9eed050ace2886c1109a',
   'category': 'dimension_value',
   'confidence': '0.94'}],
 'units': [{'term_id': 'term_6b2